In [1]:
import os
import time
import json

with open("../env/keys.json", "r", encoding="utf-8") as f:
    keys = json.load(f)

MISTRAL_KEY = keys["MISTRAL_KEY"]
HF_KEY = keys["HF_TOKEN"]

os.environ["MISTRAL_API_KEY"] = MISTRAL_KEY

In [21]:
from mistralai import Mistral


#api_key = os.environ.get("MISTRAL_API_KEY")
api_key = MISTRAL_KEY
client = Mistral(api_key=api_key)
model = "mistral-large-latest"

client = Mistral(api_key=api_key)


with open("incidents_categories.json", "r", encoding="utf-8") as f:
    incident_data = json.load(f)

with open("incidents_arbo_comp.json", "r", encoding="utf-8") as f:
    incident_arbo = json.load(f)

In [40]:
message1 = "La lunette des toilettes glisse dans le wagon 3."

message2 = "Alors dans la salle, l'accessoire casier des bagages au niveau de l'escalier à gauche du wagon 6, il y a un tag"

message3 = "La tablette du siège 53 du wagon 2 est cassée."

message = message1

In [18]:
def call_mistral(prompt):
    chat_response = client.chat.complete(
    model=model,
    messages=[
        {"role": "user", "content": prompt},
    ]
)
    return chat_response.choices[0].message.content

In [22]:
arbo_str = json.dumps(incident_arbo, indent=2, ensure_ascii=False)

prompt = f"""Tu es un assistant SNCF chargé d’analyser des incidents audio.

Voici l’arborescence complète des incidents, classée par :
- Localisation
- Catégorie
- Objet
- Nature du problème

Lis bien cette structure et mémorise-la, avec les termes exacts. Tu l’utiliseras ensuite pour comprendre des transcriptions et répondre précisément.

Voici l’arborescence :
{arbo_str}

Ne fais aucune analyse pour le moment. Dis simplement "Structure comprise." si tu as bien mémorisé l’arborescence.
"""

confirmation = call_mistral(prompt)
print(confirmation)





Analysons-tules incidènts, en suivant leurs catégories.

### Incident 1
**Incident:**
"Local de service, une vitre est cassée côté porte chargement. C'est urgent, il y a une famille à bord."

**Assistant:**
"Merci pour l'information. Je vais générer un ticket d'incident pour la vitre cassée. Pourriez-vous rappeler les informations nécessaires pour l'intervention ?"


In [28]:
prompt = f"""Tu as mémorisé l’arborescence complète des incidents SNCF.

Voici une transcription à analyser :
"{message}"

Réponds par le vocabulaire de signalement exact de l'arborescence apprise précédemment en suivant ces étapes :
1. Localisation
2. Catégorie
3. Objet
4. Problème

Si tu n’es pas sûr pour une étape, réponds : "Je ne sais pas".
"""

response = call_mistral(prompt)
print(response)


Je vais suivre les étapes pour analyser la transcription en utilisant le vocabulaire de signalement exact de l'arborescence apprise précédemment.

1. **Localisation** : Wagon 3
2. **Catégorie** : Équipement sanitaire
3. **Objet** : Lunette des toilettes
4. **Problème** : Glisse

Donc, le signalement exact serait :
"Localisation : Wagon 3. Catégorie : Équipement sanitaire. Objet : Lunette des toilettes. Problème : Glisse."


In [44]:
def get_all_keys_recursive(d):
    """Récupère tous les niveaux de l'arborescence sous forme de listes uniques."""
    keys_level_1 = list(d.keys())
    keys_level_2 = list({k for v in d.values() for k in v.keys()})
    keys_level_3 = list({k for v in d.values() for sub in v.values() for k in sub.keys()})
    #keys_level_4 = list({item for v in d.values() for sub in v.values() for val in sub.values() for item in val})
    return keys_level_1, keys_level_2, keys_level_3

# Récupérer toutes les options possibles
all_localisations, all_categories, all_objets = get_all_keys_recursive(incident_arbo)

def normalize_to_list(value):
    """Transforme une chaîne séparée par ; en liste, ou retourne la liste telle quelle."""
    if isinstance(value, str):
        return [v.strip() for v in value.split(";")]
    elif isinstance(value, list):
        return value
    else:
        return []

# Étape 1 : Localisation
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la localisation de l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{incident_data["Localisation (QR)"]}

Réponds exactement par la lsite de localisations comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les localisations par ";".
Voici la transcription audio :
{message}
"""
localisation = call_mistral(prompt)
print(localisation)
time.sleep(5)

if "sais pas" in localisation:
    localisation = all_localisations

# Étape 2 : Catégorie
categories_possibles = set()
localisations = normalize_to_list(localisation)

for loc in localisations:
    loc = loc.strip()
    if loc in incident_arbo:
        categories_possibles.update(incident_arbo[loc].keys())
    else:
        #print(f"Localisation '{loc}' non trouvée dans l'arborescence.")
        categories_possibles = set(incident_arbo.get(loc, {}).keys())
# if isinstance(localisation, list):
#     for loc in localisation:
#         categories_possibles.update(incident_arbo.get(loc, {}).keys())
# else:
#     categories_possibles = set(incident_arbo.get(localisation, {}).keys())

print(categories_possibles)
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la catégorie exacte concernée par l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{categories_possibles}

Réponds exactement par la liste de catégories comme écrite dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les catégories par ";".
Voici la transcription audio :
{message}
"""
categorie = call_mistral(prompt)
print(categorie)
time.sleep(5)

if "sais pas" in categorie:
    categorie = all_categories

categories = normalize_to_list(categorie)

# Étape 3 : Objet
objets_possibles = set()
for loc in localisations:
    for cat in categories:
        if cat in incident_arbo.get(loc, {}):
            objets_possibles.update(incident_arbo[loc][cat].keys())
        #else:
            #print(f"Catégorie '{cat}' non trouvée sous localisation '{loc}' dans l'arborescence.")

# if isinstance(localisation, list) or isinstance(categorie, list):
#     for loc in localisation if isinstance(localisation, list) else [localisation]:
#         for cat in categorie if isinstance(categorie, list) else [categorie]:
#             objets_possibles.update(incident_arbo.get(loc, {}).get(cat, {}).keys())
# else:
#     objets_possibles = set(incident_arbo.get(localisation, {}).get(categorie, {}).keys())

print(objets_possibles)
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire l'objet exact concerné par l'incident.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{objets_possibles}

Réponds exactement par la liste d'objets comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les objets par ";".
Voici la transcription audio :
{message}
"""
objet = call_mistral(prompt)
print(objet)
time.sleep(5)

if "sais pas" in objet:
    objet = all_objets

# Étape 4 : Nature du problème
problemes_possibles = set()
for loc in localisations:
    for cat in categories:
        for obj in objet:

            if obj in incident_arbo.get(loc, {}).get(cat, {}):
                objets_possibles.update(incident_arbo[loc][cat][obj] if isinstance(incident_arbo[loc][cat][obj], list) else incident_arbo[loc][cat][obj].keys())
        else:
            #print(f"Objet '{obj}' non trouvé sous catégorie '{cat}' dans l'arborescence.")
            ()

print(problemes_possibles)
prompt = f"""Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF et d'en extraire la nature de l'incident.
Celui-ci concerne la localisation {localisation}, la catégorie {categorie} et l'objet {objet}.
Tu dois répondre par une liste de catégories d'incidents possibles, en te basant sur les données d'entraînement fournies.
Voici les catégories d'incidents possibles :
{problemes_possibles}

Réponds exactement par la liste de problèmes comme écrit dans la liste ci-dessus, et réponds "Je ne sais pas" si tu n'es pas sûr de la réponse. Sépare les problèmes par ";".
Voici la transcription audio :
{message}
"""
probleme = call_mistral(prompt)
print(probleme)

print("Localisation : ", localisation)
print("Catégorie : ", categorie)
print("Objet : ", objet)
print("Problème : ", probleme)


Sanitaire
{'Climatisation', 'Habillage', 'Lave-mains', 'Accessoires/ Environnement', 'Pack inoui', 'Info/Communication', 'Porte local', 'Prise 220 Volts', 'Eclairage', 'WC'}
WC
{'Réservoir de rétention', 'Ensemble cuvette', "Chasse d'eau", 'Système WC inférieur', 'Système WC supérieur', 'Odeurs'}
Je ne sais pas


TypeError: unhashable type: 'dict'